**Created by Justin Cooke on April 27th, 2026**

Purpose of this script is to calculate a GEM in the region of the LC from the HYCOM data to investigate why do the larger scales that exist in the model result in higher variance.

We hope to find what's the depth of the strongest $N^2$ across the jet and what is that value?

In [47]:
# Import modules

import warnings
warnings.filterwarnings('ignore')

# Sci computing
import numpy as np
import scipy as sp
import seawater as sw
import scipy.sparse.linalg as sla

# Parallel comupting
from dask.distributed import Client, LocalCluster
from dask.diagnostics import ProgressBar

# For Data
import netCDF4 as nc
import xarray as xr

# Plotting stuff
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.gridspec as grdspc
import cmocean as cm

# Importing bathymetry (tiff)
import rasterio as geo

# Gen stuff
from datetime import date
today = date.today()
import glob

In [48]:
# First we need to define a function dpth which is converts pressure to depth in m

def dpth(pres_dbar,lat_deg):
    x = np.sin(np.radians(lat_deg))**2
    g = 9.780318 * (1.0 + (5.2788e-3 + 2.36e-5 * x) * x)

    depth_in_m = (((-1.82e-15 * pres_dbar + 2.279e-10) * pres_dbar - 2.2512e-5) * pres_dbar + 9.72659) * pres_dbar / g

    return depth_in_m

def prs(depth_meters, latitude_deg, tol=0.001):
    # Iteratively compute pressure [dbar] from depth [m] and latitude [deg] 

    # Parameters
    # depth_meters : float or np.ndarray
        # Depth in meters
    # latitude_deg : float or np.ndarray
        # Latitude degrees north (-90 to 90)
    # tol          : float, optional

    # Returns
    # pressure : np.ndarray 
        # Pressure in decibar (dbar)
    # iterations : int
        # Number of iterations used to converge

    # Convert inputs to arrays

    depth_meters = np.atleast_1d(depth_meters).astype(float)
    latitude_deg = np.atleast_1d(latitude_deg).astype(float)

    # Broadcast latitude to depth shape if needed
    if latitude_deg.size == 1:
        latitude_deg = np.full_like(depth_meters,latitude_deg)
    elif latitude_deg.shape != depth_meters.shape:
        if latitude_deg.shape[0] == depth_meters.shape[1]:
            latitude_deg = np.tile(latitude_deg, (depth_meters.shape[0],1))
        else: 
            raise ValueError("Latitude and Depth must have compatible dimensions")
        
    # Initialization
    pressure = 1.01 * depth_meters
    converged = False
    max_iters = 20
    iters = 1

    while not converged and iters <= max_iters:
        d = dpth(pressure,latitude_deg)
        new_pressure = pressure + (depth_meters - d) * 1.01
        delta = np.abs(new_pressure - pressure)

        if np.max(delta) < tol:
            converged = True

        pressure = new_pressure
        iters += 1

    if not converged:
        pressure[:] = np.nan

    return pressure.squeeze()

In [49]:
# For plotting; have the bathymetry of the gulf

ds_gulf = geo.open('/home/justin_cooke_uri_edu/DeepCyclones/gulf.tiff')
#ds_gulf = geo.open('../gulf.tiff')
img = ds_gulf.read(1)

(gm,gn) = np.shape(img)
gulf_lon = np.linspace(-90,-82,gn)
gulf_lat = np.linspace(22,28,gm)

# To get the aspect ratio right
ymid = np.mean(gulf_lat)
ymid_rad = ymid*np.pi/180

# For plotting: labels

lon_xticks = np.array((-90,-89,-88,-87,-86,-85,-84,-83))
lon_xticklbls = ['$90^\circ$W','$89^\circ$W','$88^\circ$W','$87^\circ$W','$86^\circ$W','$85^\circ$W','$84^\circ$W','$83^\circ$W']

lat_yticks = np.array((22,23,24,25,26,27,28))
lat_yticklbls = ['$22^\circ$N','$23^\circ$N','$24^\circ$N','$25^\circ$N','$26^\circ$N','$27^\circ$N','$28^\circ$N']

In [57]:
ds_theta = xr.load_dataset('./hycom_data/hycom_temp_slice86W.nc')
ds_sal = xr.load_dataset('./hycom_data/hycom_sal_slice86W.nc')
ds_latslice = xr.load_dataset('./hycom_data/hycom_lat_slice86W.nc')

In [ ]:
# Load using XArray the lat lon and depth data first

ds_latlon = xr.open_dataset("./hycom_data/hycom_latlon.nc")
ds_depth = xr.open_dataset("./hycom_data/hycom_depth.nc")

# Now get the lat, lon, and depth for 22N to 28N and -90W to -83W and depth down to 2000m
ds_lon = ds_latlon['Longitude'][:]
ds_lat = ds_latlon['Latitude'][:]
ds_z = ds_depth['Depth'][:] 

# finding the index in lon array that corresponds to 90W
ind90 = list(np.where(ds_lon >= -90)) 
ind83 = list(np.where(ds_lon >= -83))
nlon90 = ind90[0][0]
nlon83 = ind83[0][0] + 1
lon = ds_lon[nlon90:nlon83]

# finding the indices in lat array that correspond to 22W and 28W to only grab data from this region
ind22 = list(np.where(ds_lat >= 22))
nlat22 = ind22[0][0] 
ind28 = list(np.where(ds_lat >= 28))
nlat28 = ind28[0][0] + 1
lat = ds_lat[nlat22:nlat28]

ind2k = list(np.where(ds_z >= 2000))
n2k = ind2k[0][0] + 1
depth = ds_z[:n2k]

In [ ]:
# Next, we can load the potential temperature and salinity

ds_theta = xr.load_dataset("./hycom_data/hycom_temp.nc", chunks={'MT': 540, 'Depth': 13, 'Latitude': 83, 'Longitude': 88})
ds_sal = xr.load_dataset("./hycom_data/hycom_sal.nc", chunks={'MT': 540, 'Depth': 13, 'Latitude': 83, 'Longitude': 88})

ds_ssh = xr.load_dataset("./hycom_data/hycom_ssh_filtered.nc", chunks={'MT': 540, 'Latitude': 83, 'Longitude': 88})

In [ ]:
# Now, we will mask all points that do not reach 2000 meters depth

Nt,Nz,_,_ = ds_theta['temperature'].shape

# Lazily load potential temp and salinity
theta = ds_theta['temperature']
sal = ds_sal['salinity']

# Assign coordinates to the lat, lon, and depth dimensions
theta['Longitude'] = lon
theta['Latitude'] = lat
theta['Depth'] = depth
theta['MT'] = ds_theta['MT']

sal['Longitude'] = lon
sal['Latitude'] = lat
sal['Depth'] = depth
sal['MT'] = ds_sal['MT']


# Find the points that have valid points at each depth over all lat lon
depth_mask = theta.notnull().any(dim='MT')

# Here we are multiplying our boolean by depth and taking this at the last depth level (the maximum)
deepest_valid_depth = (depth_mask * depth).max(dim='Depth')

# This is our mask for points that reach at least 2000m
has_2000m = deepest_valid_depth >= 2000

# Now we are applying our mask
masked_theta = theta.where(has_2000m)
masked_sal = sal.where(has_2000m)

# We are storing the masked potential temperature 
theta['masked'] = masked_theta
sal['masked'] = masked_sal

# Assign coordinates to the lat, lon, and depth dimensions
theta['masked']['Longitude'] = lon
theta['masked']['Latitude'] = lat
theta['masked']['Depth'] = depth
theta['masked']['MT'] = ds_theta['MT']

sal['masked']['Longitude'] = lon
sal['masked']['Latitude'] = lat
sal['masked']['Depth'] = depth
sal['masked']['MT'] = ds_sal['MT']

In [ ]:
# Converting depth to pressure (dbar)

# Need to create an mxn array where m = [depth] and n = [latitude] to be used in prs kinda like meshgrid
depth_2d, lat_2d = xr.broadcast(depth,lat)

# make depth and lat lazy
depth_2d = depth_2d.chunk({'Depth': 13, 'Latitude': 83})
lat_2d = lat_2d.chunk({'Depth': 13, 'Latitude': 83})

# This wrapper (apply_ufunc) allows converting depth to pressure
# pressure is the name of our output variable
pressure = xr.apply_ufunc( 
    prs, # the actual function we are calling
    depth_2d, # input one which is our mxn depth array
    lat_2d, # input two which is our mxn latitude array
    input_core_dims=[['Depth', 'Latitude'], ['Depth', 'Latitude']], # input and output explicitly tells xarray both inputs and outputs share the same 2d struct
    output_core_dims=[['Depth', 'Latitude']],
    dask='parallelized', # executes lazily with Dask
    vectorize=True, # ensure pres is applied elementwise across the 2D arrays
    output_dtypes=[float], # ensure consistent output type
    dask_gufunc_kwargs={'allow_rechunk': True},
)

# Zero out top row
pressure[0,:] = 0.0

# Want to create repeating matrices of pressure
pressure_full = pressure.expand_dims({
    'Longitude': theta['Longitude'],
    'MT': theta['MT']
}).transpose('MT', 'Depth', 'Latitude', 'Longitude')

pressure_full_masked = pressure_full.where(has_2000m)

# Ref pressure now
pref = xr.zeros_like(pressure_full)
pref_masked = pref.where(has_2000m)

pressure_full_masked = pressure_full.chunk({'MT': 540, 'Depth': 13, 'Latitude': 83, 'Longitude': 88})
pref_masked = pref_masked.chunk({'MT': 540, 'Depth': 13, 'Latitude': 83, 'Longitude': 88})

In [ ]:
sal_86 = sal['masked'].isel(Longitude=100,Latitude=slice(None, None, 4))
sal_86_mean = xr.DataArray.mean(sal_86,dim='MT',skipna=True)
theta_86 = theta['masked'].isel(Longitude=100,Latitude=slice(None, None, 4))
sal_86_mean

plt.contourf(sal_86_mean)

In [ ]:
# Now we can handle converting potential temp to temp

this_theta = theta['masked']
this_sal = sal['masked']

temperature = xr.apply_ufunc(
    sw.eos80.temp,
    this_sal,
    this_theta,
    pressure_full_masked,
    pref_masked,
    input_core_dims=[['MT','Depth','Latitude','Longitude']]*4,
    output_core_dims=[['MT','Depth','Latitude','Longitude']],
    dask='parallelized',
    vectorize=True,
    output_dtypes=[float],
    dask_gufunc_kwargs={'allow_rechunk': True}
)

temp_masked = temperature.where(has_2000m)
temp_masked = temp_masked.chunk({'MT': 540, 'Depth': 13, 'Latitude': 83, 'Longitude': 88})
theta['masked_temp'] = temp_masked

theta['masked_temp']['Longitude'] = lon
theta['masked_temp']['Latitude'] = lat
theta['masked_temp']['Depth'] = depth
theta['masked_temp']['MT'] = ds_theta['MT'][0:Nt_1]

In [ ]:
bfrq = xr.apply_ufunc(
    sw.bfrq,
    this_sal,
    this_theta,
    pressure_full_masked
)

In [ ]:
theta['masked_temp']